# Stage1 公共分支去重：配对 0bp 原生账户

先执行来源／Qlib DatasetH 和原账户路径复现门，再拟合原版与去重版并各自结算 0bp Tiered 费用。

In [1]:
import torch
import hashlib,json,os,pathlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.stage1_domain_heads import _read_source,_new_model
from src.workflow.minute_pit_allocator import _primary_dataset,_auxiliary_dataset
from src.model.minute_pit_allocator import MinutePitAllocator,TARGETS,_flat_index
from src.backtest.stage1_domain_heads_raw import replay_stage1_raw
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
root=pathlib.Path.cwd();card=root/'research/stage1_common_return_dedup_0bp_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal']
for path,digest in plan['code_sha256'].items():assert sha(root/path)==digest
source_plan_path=root/'research/stage1_domain_heads_20261018/plan.json';domain=root/'research/stage1_domain_heads_20261018';src=plan['source']
assert sha(source_plan_path)==src['matched_account_plan_sha256'] and sha(domain/'summary.json')==src['matched_account_summary_sha256'] and sha(domain/'verification_manifest.json')==src['matched_account_manifest_sha256']
assert sha(root/src['preflight_card']/'summary.json')==src['preflight_summary_sha256'] and sha(root/src['training_card']/'summary.json')==src['training_summary_sha256']
original_plan=json.loads(source_plan_path.read_text());assert original_plan['tariff']==plan['tariff'] and original_plan['accounts']['expected_sessions']==plan['accounts']['expected_sessions']==944
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/original_plan['provider_init']),region='us',kernels=1)
train_source,train_audit=_read_source(root,src['training_source'])
raw_spec=src['raw_reference_source'];reference_names=['native/raw_SOXX_0','native/raw_strong9097_0','native/raw_candidate27baa_0'];subset={**raw_spec,'artifacts':reference_names,'artifact_sha256':{k:raw_spec['artifact_sha256'][k] for k in reference_names}}
raw_reference,raw_audit=_read_source(root,subset)
manifest=json.loads((domain/'verification_manifest.json').read_text())
entry={item['path'].split('/artifacts/',1)[1]:item for item in manifest['entries'] if '/artifacts/' in item['path']};loaded={}
needed=['inputs/frame','inputs/market','inputs/raw_points','inputs/schedule','inputs/controller','inputs/selection','inputs/recovery_spec','inputs/action_spec','inputs/guard','paths/A_active_research_replay_0','native/A_active_research_replay_0']
with R.uri_context(str(domain/'qlib_recorder')):
 rec=R.get_recorder(experiment_id='283510277269468590',recorder_id='6e227278b7a64eafbcf34b965e16d2b9');assert rec.status=='FINISHED'
 for name in needed:
  path=domain/'qlib_recorder'/'283510277269468590'/'6e227278b7a64eafbcf34b965e16d2b9'/'artifacts'/name
  assert sha(path)==entry[name]['sha256'];loaded[name]=rec.load_object(name)
first=train_source['models/base_outer_71'];assert first.seed==plan['training']['seed']==71 and len(first.base_features)==39 and first.spec.epochs==300
inputs={'frame':train_source['inputs/daily_frame'],'features':train_source['inputs/features'],'common':train_source['inputs/common'],'timing':train_source['inputs/timing'],'spec':first.spec}
base=_primary_dataset(inputs,train_end=str(first.train_end.date()),predict_bounds=train_source['inputs/plan']['outer_window'])
aux_base=_auxiliary_dataset(train_source['inputs/auxiliary']['outer'],str(first.train_end.date()))
rows=_flat_index(base.prepare('train',col_set=['feature','label','common','meta']));maturity=pd.to_datetime(rows[('meta','label_available_at')]);eligible=maturity.notna()&maturity.lt(first.cutoff)&(rows.index<=first.train_end)&np.isfinite(rows['label'][list(TARGETS)].to_numpy(float)).all(axis=1);actual=rows.loc[eligible]
assert len(actual)==plan['training']['mature_primary_rows']==3014 and len(aux_base.prepare('train',col_set=['feature','label','meta']))==plan['training']['auxiliary_rows']==20741
medians=plan['intervention']['training_medians'];assert all(abs(float(actual['common'][k].median())-v)<1e-12 for k,v in medians.items())
masked_common=inputs['common'].copy(deep=True);masked_aux=train_source['inputs/auxiliary']['outer'].copy(deep=True)
for name,value in medians.items():masked_common[name]=value;masked_aux[name]=value
candidate_dataset=_primary_dataset({**inputs,'common':masked_common},train_end=str(first.train_end.date()),predict_bounds=train_source['inputs/plan']['outer_window']);candidate_aux=_auxiliary_dataset(masked_aux,str(first.train_end.date()))
for segment in ('train','predict'):
 a=base.prepare(segment,col_set=['feature','label','common','meta']);b=candidate_dataset.prepare(segment,col_set=['feature','label','common','meta'])
 assert len(a)==len(b)
 for group in ('feature','label','meta'):pd.testing.assert_frame_equal(a[group],b[group],check_exact=True)
 rest=[x for x in a['common'].columns if x not in medians];pd.testing.assert_frame_equal(a['common'][rest],b['common'][rest],check_exact=True)
 assert all(b['common'][name].eq(value).all() for name,value in medians.items())
assert len(base.prepare('predict',col_set='feature'))==plan['training']['predict_rows']==946
frame=loaded['inputs/frame'];market=loaded['inputs/market'];points=loaded['inputs/raw_points'];schedule=loaded['inputs/schedule'];guard=loaded['inputs/guard'];controller=loaded['inputs/controller'];selection=loaded['inputs/selection'];action=Stage1OpenActionValueSpec(root=root,**loaded['inputs/action_spec']);recovery=IntradayRecoveryDirectAllocationSpec(root=root,**loaded['inputs/recovery_spec'])
assert len(frame)==len(market)==944 and len(points)==3778
pd.testing.assert_frame_equal(frame[['signal_date','trade_date','exit_date']],market[['signal_date','trade_date','exit_date']])
execution={'initial_cash':100000,'tariff':plan['tariff']}
a_replay=replay_stage1_raw(frame.copy(deep=True),point_prices=points,schedule=schedule,plan=execution,controller=controller,recovery_spec=recovery,action_spec=action,selection=selection,slippage_bps=0)
saved_a=loaded['paths/A_active_research_replay_0']
for key in ('replay','orders','points','targets','trace'):pd.testing.assert_frame_equal(a_replay[key],saved_a[key],check_exact=True)
preflight={'status':'original_Stage1_A0_current_code_path_exact_parity_and_masked_DatasetH_contract_passed','plan_sha256':sha(card/'plan.json'),'source_training_objects':len(src['training_source']['artifacts']),'source_raw_reference_objects':len(reference_names),'source_matched_account_objects':len(needed),'mature_training_rows':len(actual),'auxiliary_rows':20741,'predict_rows':946,'mask_medians':medians,'A0_path_exact_fields':['replay','orders','points','targets','trace'],'new_model_fits':0,'new_native_accounts':0,'A0_replay_paths':1}
(card/'preflight_result.json').write_text(json.dumps(preflight,ensure_ascii=False,indent=2)+'\n');print(json.dumps(preflight,ensure_ascii=False,indent=2))


{
  "status": "original_Stage1_A0_current_code_path_exact_parity_and_masked_DatasetH_contract_passed",
  "plan_sha256": "d33c5ea855bd06fb8a41911b6bace14b8ab0a0a0911db7325f5b730c96399d14",
  "source_training_objects": 10,
  "source_raw_reference_objects": 3,
  "source_matched_account_objects": 11,
  "mature_training_rows": 3014,
  "auxiliary_rows": 20741,
  "predict_rows": 946,
  "mask_medians": {
    "mom5": 0.006551146507263184,
    "mom20": 0.019736766815185547,
    "mom60": 0.05451840162277222
  },
  "A0_path_exact_fields": [
    "replay",
    "orders",
    "points",
    "targets",
    "trace"
  ],
  "new_model_fits": 0,
  "new_native_accounts": 0,
  "A0_replay_paths": 1
}


In [2]:
import torch
import hashlib,json,os,pathlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.stage1_domain_heads import _read_source,_new_model
from src.workflow.minute_pit_allocator import _primary_dataset,_auxiliary_dataset
from src.model.minute_pit_allocator import MinutePitAllocator,TARGETS,_flat_index
from src.backtest.stage1_domain_heads_raw import replay_stage1_raw
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
root=pathlib.Path.cwd();card=root/'research/stage1_common_return_dedup_0bp_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal']
saved_preflight=json.loads((card/'preflight_result.json').read_text())
assert saved_preflight['status']=='original_Stage1_A0_current_code_path_exact_parity_and_masked_DatasetH_contract_passed' and saved_preflight['plan_sha256']==sha(card/'plan.json')
for path,digest in plan['code_sha256'].items():assert sha(root/path)==digest
source_plan_path=root/'research/stage1_domain_heads_20261018/plan.json';domain=root/'research/stage1_domain_heads_20261018';src=plan['source']
assert sha(source_plan_path)==src['matched_account_plan_sha256'] and sha(domain/'summary.json')==src['matched_account_summary_sha256'] and sha(domain/'verification_manifest.json')==src['matched_account_manifest_sha256']
assert sha(root/src['preflight_card']/'summary.json')==src['preflight_summary_sha256'] and sha(root/src['training_card']/'summary.json')==src['training_summary_sha256']
original_plan=json.loads(source_plan_path.read_text());assert original_plan['tariff']==plan['tariff'] and original_plan['accounts']['expected_sessions']==plan['accounts']['expected_sessions']==944
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/original_plan['provider_init']),region='us',kernels=1)
train_source,train_audit=_read_source(root,src['training_source'])
raw_spec=src['raw_reference_source'];reference_names=['native/raw_SOXX_0','native/raw_strong9097_0','native/raw_candidate27baa_0'];subset={**raw_spec,'artifacts':reference_names,'artifact_sha256':{k:raw_spec['artifact_sha256'][k] for k in reference_names}}
raw_reference,raw_audit=_read_source(root,subset)
manifest=json.loads((domain/'verification_manifest.json').read_text())
entry={item['path'].split('/artifacts/',1)[1]:item for item in manifest['entries'] if '/artifacts/' in item['path']};loaded={}
needed=['inputs/frame','inputs/market','inputs/raw_points','inputs/schedule','inputs/controller','inputs/selection','inputs/recovery_spec','inputs/action_spec','inputs/guard','paths/A_active_research_replay_0','native/A_active_research_replay_0']
with R.uri_context(str(domain/'qlib_recorder')):
 rec=R.get_recorder(experiment_id='283510277269468590',recorder_id='6e227278b7a64eafbcf34b965e16d2b9');assert rec.status=='FINISHED'
 for name in needed:
  path=domain/'qlib_recorder'/'283510277269468590'/'6e227278b7a64eafbcf34b965e16d2b9'/'artifacts'/name
  assert sha(path)==entry[name]['sha256'];loaded[name]=rec.load_object(name)
first=train_source['models/base_outer_71'];assert first.seed==plan['training']['seed']==71 and len(first.base_features)==39 and first.spec.epochs==300
inputs={'frame':train_source['inputs/daily_frame'],'features':train_source['inputs/features'],'common':train_source['inputs/common'],'timing':train_source['inputs/timing'],'spec':first.spec}
base=_primary_dataset(inputs,train_end=str(first.train_end.date()),predict_bounds=train_source['inputs/plan']['outer_window'])
aux_base=_auxiliary_dataset(train_source['inputs/auxiliary']['outer'],str(first.train_end.date()))
rows=_flat_index(base.prepare('train',col_set=['feature','label','common','meta']));maturity=pd.to_datetime(rows[('meta','label_available_at')]);eligible=maturity.notna()&maturity.lt(first.cutoff)&(rows.index<=first.train_end)&np.isfinite(rows['label'][list(TARGETS)].to_numpy(float)).all(axis=1);actual=rows.loc[eligible]
assert len(actual)==plan['training']['mature_primary_rows']==3014 and len(aux_base.prepare('train',col_set=['feature','label','meta']))==plan['training']['auxiliary_rows']==20741
medians=plan['intervention']['training_medians'];assert all(abs(float(actual['common'][k].median())-v)<1e-12 for k,v in medians.items())
masked_common=inputs['common'].copy(deep=True);masked_aux=train_source['inputs/auxiliary']['outer'].copy(deep=True)
for name,value in medians.items():masked_common[name]=value;masked_aux[name]=value
candidate_dataset=_primary_dataset({**inputs,'common':masked_common},train_end=str(first.train_end.date()),predict_bounds=train_source['inputs/plan']['outer_window']);candidate_aux=_auxiliary_dataset(masked_aux,str(first.train_end.date()))
for segment in ('train','predict'):
 a=base.prepare(segment,col_set=['feature','label','common','meta']);b=candidate_dataset.prepare(segment,col_set=['feature','label','common','meta'])
 assert len(a)==len(b)
 for group in ('feature','label','meta'):pd.testing.assert_frame_equal(a[group],b[group],check_exact=True)
 rest=[x for x in a['common'].columns if x not in medians];pd.testing.assert_frame_equal(a['common'][rest],b['common'][rest],check_exact=True)
 assert all(b['common'][name].eq(value).all() for name,value in medians.items())
assert len(base.prepare('predict',col_set='feature'))==plan['training']['predict_rows']==946
frame=loaded['inputs/frame'];market=loaded['inputs/market'];points=loaded['inputs/raw_points'];schedule=loaded['inputs/schedule'];guard=loaded['inputs/guard'];controller=loaded['inputs/controller'];selection=loaded['inputs/selection'];action=Stage1OpenActionValueSpec(root=root,**loaded['inputs/action_spec']);recovery=IntradayRecoveryDirectAllocationSpec(root=root,**loaded['inputs/recovery_spec'])
assert len(frame)==len(market)==944 and len(points)==3778
pd.testing.assert_frame_equal(frame[['signal_date','trade_date','exit_date']],market[['signal_date','trade_date','exit_date']])
execution={'initial_cash':100000,'tariff':plan['tariff']}
from src.workflow.minute_pit_allocator import _prediction_dates,_ensemble
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_domain_heads import _provider
from src.backtest.native_raw_corporate import native_raw_validate
from src.analysis.stage1_domain_heads import analyze_paired_accounts
from src.analysis.integrated_allocation import paired_metrics,stationary_bootstrap_excess,validate_replay
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
from qlib.workflow.recorder import Recorder
import shlex
state_path=card/'run_state.json';state={'status':'fit_started','plan_sha256':sha(card/'plan.json'),'source_objects_read_after_handoff':len(src['training_source']['artifacts'])+len(reference_names)+len(needed),'fit_attempts':[],'fits_done':[],'path_attempts':[],'paths_done':[],'native_attempts':[],'native_done':[],'bootstrap_attempts':0}
state_path.write_text(json.dumps(state,indent=2)+'\n')
b_model=_new_model(first,MinutePitAllocator,root);c_model=_new_model(first,MinutePitAllocator,root)
b_init=b_model._initialize();c_init=c_model._initialize()
for name,value in b_init.state_dict().items():assert torch.equal(value,c_init.state_dict()[name])
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid,eid=rec.id,str(rec.experiment_id);state.update(recorder_id=rid,experiment_id=eid);state_path.write_text(json.dumps(state,indent=2)+'\n')
 base.config(dump_all=True,recursive=True);aux_base.config(dump_all=True,recursive=True);candidate_dataset.config(dump_all=True,recursive=True);candidate_aux.config(dump_all=True,recursive=True)
 rec.save_objects(artifact_path='inputs',plan=plan,preflight=saved_preflight,baseline_dataset=base,candidate_dataset=candidate_dataset,baseline_auxiliary=aux_base,candidate_auxiliary=candidate_aux)
 predictions={}
 for family,model,dataset,aux in [('B_original55',b_model,base,aux_base),('C_common_dedup',c_model,candidate_dataset,candidate_aux)]:
  if len(state['fit_attempts'])>=plan['budget']['new_model_fit_attempts_max']:raise ValueError('blocked: fit budget consumed')
  state['fit_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  model.fit(dataset,auxiliary_dataset=aux)
  assert model.audit['training_rows']==3014 and model.audit['auxiliary_rows']==20741 and model.audit['epochs']==300
  assert model.primary_scaler==first.primary_scaler
  if family=='B_original55':assert model.common_scaler==first.common_scaler
  else:
   for k in model.common_scaler:
    for name in first.common_scaler[k]:
     if name not in medians:assert model.common_scaler[k][name]==first.common_scaler[k][name]
  state['fits_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  prediction=_prediction_dates(model.predict(dataset));assert len(prediction)==946
  if family=='B_original55':
   saved=train_source['analysis/outer_predictions'][('base',71)]
   pd.testing.assert_index_equal(prediction.index,saved.index)
   np.testing.assert_allclose(prediction.to_numpy(float),saved.to_numpy(float),rtol=1e-6,atol=1e-7)
   state['B_saved_prediction_parity']=True;state_path.write_text(json.dumps(state,indent=2)+'\n')
  predictions[family]=prediction;rec.save_objects(artifact_path='models',**{family:model});rec.save_objects(artifact_path='predictions',**{family:prediction})
  rec.log_metrics(completed_fits=len(state['fits_done']))
 dates=pd.DatetimeIndex(market.signal_date);returns=market.set_index('signal_date')[['soxl_return','soxx_return']]
 targets={family:_smooth_weights(pred.reindex(dates),spec=first.spec,candidate=first.candidate,asset_returns=returns) for family,pred in predictions.items()}
 for family,target in targets.items():
  pd.testing.assert_index_equal(target.index,dates)
  assert list(target)==['soxl_weight','soxx_weight','cash_weight','tail_probability','alpha_probability'] and np.isfinite(target.to_numpy(float)).all()
  rec.save_objects(artifact_path='targets',**{family:target})
state['status']='model_and_target_barrier_passed';state_path.write_text(json.dumps(state,indent=2)+'\n')
# The native validator can change Qlib's active provider; use the saved Recorder directly.
rec.start_run()
try:
 provider,provider_manifest=_provider(root,{'points':points,'guard':guard,'plan':{'provider_output':'data/qlib/'+plan['experiment']}})
 rec.save_objects(artifact_path='inputs',native_provider_manifest=provider_manifest)
 books={('A_active_research_replay',0):loaded['native/A_active_research_replay_0'],('SOXX',0):raw_reference['native/raw_SOXX_0'],('strong9097',0):raw_reference['native/raw_strong9097_0'],('additional27baa',0):raw_reference['native/raw_candidate27baa_0']}
 for family in ('B_original55','C_common_dedup'):
  if len(state['path_attempts'])>=2:raise ValueError('blocked: path budget consumed')
  state['path_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  candidate_frame=frame.copy(deep=True)
  for name in targets[family]:candidate_frame[name]=targets[family][name].to_numpy(copy=True)
  path=replay_stage1_raw(candidate_frame,point_prices=points,schedule=schedule,plan=execution,controller=controller,recovery_spec=recovery,action_spec=action,selection=selection,slippage_bps=0)
  assert len(path['replay'])==944 and len(path['points'])==3778
  rec.save_objects(artifact_path='paths',**{family:path});state['paths_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  if len(state['native_attempts'])>=2:raise ValueError('blocked: native budget consumed')
  state['native_attempts'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
  book=native_raw_validate(path,provider_uri=provider,point_prices=points,schedule=schedule,tariff=plan['tariff'],cost=0,initial_cash=100000)
  assert book['audit']['status']=='passed'
  pd.testing.assert_frame_equal(book['authority'][['signal_date','trade_date','exit_date']].reset_index(drop=True),frame[['signal_date','trade_date','exit_date']].reset_index(drop=True))
  rec.save_objects(artifact_path='native',**{family:book});books[(family,0)]=book;state['native_done'].append(family);state_path.write_text(json.dumps(state,indent=2)+'\n')
 analysis=analyze_paired_accounts(books,train_source['inputs/daily_frame'],plan,candidate_family='C_common_dedup',required_comparators=('SOXX',),expected_families=tuple(plan['accounts']['families']))
 bpath=validate_replay(books[('B_original55',0)]['authority']);cpath=validate_replay(books[('C_common_dedup',0)]['authority'])
 matched=paired_metrics(cpath,bpath);ci_args={k:plan['gates']['paired_uncertainty'][k] for k in ('mean_block','resamples','alpha','seed')}
 state['bootstrap_attempts']=2;state_path.write_text(json.dumps(state,indent=2)+'\n')
 matched_uncertainty=stationary_bootstrap_excess(cpath,bpath,**ci_args)
 rec.save_objects(artifact_path='analysis',metrics=analysis['metrics'],paired_comparison=analysis['paired_comparison'],annual_metrics=analysis['annual_metrics'],quarterly_metrics=analysis['quarterly_metrics'],rolling_summary=analysis['rolling_summary'],uncertainty=analysis['uncertainty'],rolling=analysis['rolling'],gates=analysis['gates'],exposure_summary=analysis['exposure_summary'],matched_B_C=matched,matched_B_C_uncertainty=matched_uncertainty)
 rec.log_metrics(completed_native_accounts=2,candidate_SOXX_relative_wealth=float(analysis['metrics'].loc[analysis['metrics'].family.eq('C_common_dedup'),'relative_wealth'].iloc[0]),rolling_positive_fraction=float(analysis['rolling_summary'].loc[analysis['rolling_summary'].family.eq('C_common_dedup'),'positive_fraction'].iloc[0]))
 rec.end_run(status=Recorder.STATUS_FI)
except BaseException:
 rec.end_run(status=Recorder.STATUS_FA)
 raise
state['status']='finished';state_path.write_text(json.dumps(state,indent=2)+'\n')
metric=analysis['metrics'].loc[analysis['metrics'].family.eq('C_common_dedup')].iloc[0];roll=analysis['rolling_summary'].loc[analysis['rolling_summary'].family.eq('C_common_dedup')].iloc[0];soxx_ci=analysis['uncertainty'].iloc[0]
coarse_keys=['net_relative_wealth_all_costs','sampled_drawdown_30pct','cvar_no_worse_soxx','minimum_complete_years','complete_annual_70pct','rolling126_70pct','paired_block_lower_positive']
coarse_pass=all(analysis['gates'][key] for key in coarse_keys)
def ready(value):
 if isinstance(value,dict):return {str(k):ready(v) for k,v in value.items()}
 if isinstance(value,(list,tuple)):return [ready(v) for v in value]
 if isinstance(value,np.generic):return value.item()
 return value
summary={'experiment':plan['experiment'],'decision':'conditional_Stage1_common_return_dedup_0bp_SOXX_coarse_pass_needs_2_5bp_forward' if coarse_pass else 'not_promoted_fixed_Stage1_common_return_dedup_0bp_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':'preflight_result.json','A0_exact_path_parity':True,'B_saved_prediction_parity':state.get('B_saved_prediction_parity',False),'new_model_fits':len(state['fits_done']),'new_A0_parity_paths':1,'new_candidate_paths':len(state['paths_done']),'new_native_accounts':len(state['native_done']),'new_bootstraps':state['bootstrap_attempts'],'coarse_SOXX_0bp_hard_gate_pass':coarse_pass,'gates':analysis['gates'],'candidate_SOXX_relative_wealth':float(metric.relative_wealth),'candidate_close_max_drawdown_pct':float(metric.close_max_drawdown_pct),'candidate_sampled_point_max_drawdown_pct':float(metric.sampled_point_max_drawdown_pct),'candidate_cvar5_pct':float(metric.cvar5_pct),'SOXX_cvar5_pct':float(metric.benchmark_cvar5_pct),'candidate_rolling_positive_fraction':float(roll.positive_fraction),'candidate_rolling_windows':int(roll.windows),'SOXX_bootstrap_lower_bound':float(soxx_ci.lower_bound),'matched_B_C_relative_wealth':matched['relative_wealth'],'matched_B_C_bootstrap_lower_bound':matched_uncertainty['lower_bound'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'artifact_policy':plan['artifact_policy'],'limitations':['Same 2023-2026 market history reused after many routes; no independent forward qualification.','Full-minute risk remains unqualified; 0bp pass alone cannot satisfy v2 2/5bp and forward gates.']}
summary=ready(summary);(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);settings=_settings(root);remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256'];summary['NAS_evidence']=evidence;summary['NAS_remote_hash_verified']=True;(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in summary.items() if k not in ('gates','NAS_evidence','artifact_policy','limitations')},ensure_ascii=False,indent=2));print('gates',analysis['gates'])


2 models and 2 native 0bp accounts saved; original three-cost analyzer rejected the planned 0bp-only result. See attempt1_analysis_failure.json.


In [3]:
import hashlib,json,math,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.analysis.integrated_allocation import paired_metrics,stationary_bootstrap_excess,validate_replay
from src.analysis.stage1_domain_heads import _drawdown_pct
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_common_return_dedup_0bp_20261023';plan=json.loads((card/'plan.json').read_text());repair=json.loads((card/'analysis_repair_plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert sha(card/'plan.json')==repair['original_plan_sha256']
state=json.loads((card/'run_state.json').read_text());assert state['fits_done']==['B_original55','C_common_dedup'] and state['native_done']==['B_original55','C_common_dedup'] and state['bootstrap_attempts']==0 and state['B_saved_prediction_parity']
rid,eid=repair['candidate_recorder_id'],repair['candidate_experiment_id'];assert rid==state['recorder_id'] and eid==state['experiment_id']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(str(card/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=eid,recorder_id=rid);assert rec.status=='FAILED'
 books={}
 for family in ('B_original55','C_common_dedup'):
  path=card/'qlib_recorder'/eid/rid/'artifacts/native'/family;assert sha(path)==repair['saved_native_artifact_sha256'][family]
  books[family]=rec.load_object('native/'+family);assert books[family]['audit']['status']=='passed'
domain=root/'research/stage1_domain_heads_20261018';ds=json.loads((domain/'summary.json').read_text());assert ds['recorder_id']=='6e227278b7a64eafbcf34b965e16d2b9'
with R.uri_context(str(domain/'qlib_recorder')):
 ref=R.get_recorder(experiment_id=ds['experiment_id'],recorder_id=ds['recorder_id']);assert ref.status=='FINISHED'
 path=domain/'qlib_recorder'/ds['experiment_id']/ds['recorder_id']/'artifacts/native/A_active_research_replay_0';assert sha(path)==repair['source_A0_native_sha256']
 books['A_active_research_replay']=ref.load_object('native/A_active_research_replay_0')
rawspec=plan['source']['raw_reference_source'];rawcard=root/rawspec['card'];rawsum=json.loads((rawcard/'summary.json').read_text())
with R.uri_context(str(rawcard/'qlib_recorder')):
 ref=R.get_recorder(experiment_id=rawsum['experiment_id'],recorder_id=rawspec['recorder_id']);assert ref.status=='FINISHED'
 path=rawcard/'qlib_recorder'/rawsum['experiment_id']/rawspec['recorder_id']/'artifacts/native/raw_SOXX_0';assert sha(path)==repair['source_SOXX_0_native_sha256']
 books['SOXX']=ref.load_object('native/raw_SOXX_0')
traincard=root/plan['source']['training_card'];trainsum=json.loads((traincard/'summary.json').read_text())
with R.uri_context(str(traincard/'qlib_recorder')):
 ref=R.get_recorder(experiment_id=trainsum['experiment_id'],recorder_id=trainsum['recorder_id']);assert ref.status=='FINISHED'
 calendar_source=ref.load_object('inputs/daily_frame')
calendar=pd.DatetimeIndex(pd.to_datetime(calendar_source.trade_date).dropna()).normalize().unique().sort_values();assert calendar.is_monotonic_increasing
paths={name:validate_replay(book['authority']) for name,book in books.items()};c=paths['C_common_dedup'];b=paths['B_original55'];s=paths['SOXX'];a=paths['A_active_research_replay'];assert len(c)==len(b)==len(s)==len(a)==944
for name,path in paths.items():
 assert books[name]['audit']['status']=='passed'
 pd.testing.assert_frame_equal(path[['signal_date','trade_date','exit_date']].reset_index(drop=True),s[['signal_date','trade_date','exit_date']].reset_index(drop=True))
 assert pd.DatetimeIndex(pd.to_datetime(books[name]['points'].stamp)).equals(pd.DatetimeIndex(pd.to_datetime(books['SOXX']['points'].stamp)))
relative={name:paired_metrics(path,s) for name,path in paths.items() if name!='SOXX'}
trade_dates=pd.DatetimeIndex(c.trade_date);labels=trade_dates.to_period('Y');annual=[]
for year in labels.unique():
 mask=np.asarray(labels==year);observed=trade_dates[mask];expected=calendar[calendar.to_period('Y')==year]
 complete=bool(calendar[0]<year.start_time and calendar[-1]>year.end_time and observed.equals(expected))
 annual.append({'year':str(year),'sessions':int(mask.sum()),'complete_year':complete,'relative_wealth':float(paired_metrics(c.loc[mask],s.loc[mask])['relative_wealth'])})
quarterly=[];qlabel=trade_dates.to_period('Q')
for quarter in qlabel.unique():
 mask=np.asarray(qlabel==quarter);quarterly.append({'quarter':str(quarter),'sessions':int(mask.sum()),'relative_wealth':float(paired_metrics(c.loc[mask],s.loc[mask])['relative_wealth'])})
log_excess=np.log1p(c.strategy_return.to_numpy(float))-np.log1p(s.strategy_return.to_numpy(float));rolling=pd.DataFrame({'first_trade':trade_dates[:-125].to_numpy(),'last_exit':pd.DatetimeIndex(c.exit_date)[125:].to_numpy(),'relative_log':pd.Series(log_excess).rolling(126).sum().iloc[125:].to_numpy()});assert len(rolling)==819;rolling['relative_wealth']=np.exp(rolling.relative_log)
points=books['C_common_dedup']['points'];stamps=pd.DatetimeIndex(pd.to_datetime(points.stamp));closes=pd.Series(points.nav.to_numpy(float),index=stamps).groupby(stamps.normalize()).last().reindex(trade_dates);assert closes.notna().all()
close_dd=_drawdown_pct(closes,100000);sampled_dd=_drawdown_pct(points.nav,100000);source_metrics=relative['C_common_dedup'];complete=[x for x in annual if x['complete_year']];need=math.ceil(.70*len(rolling));positive=int((rolling.relative_log>0).sum())
bootstrap=plan['gates']['paired_uncertainty'];args={k:bootstrap[k] for k in ('mean_block','resamples','alpha','seed')};soxx_ci=stationary_bootstrap_excess(c,s,**args);matched=paired_metrics(c,b);matched_ci=stationary_bootstrap_excess(c,b,**args)
gates={'SOXX_wealth':source_metrics['relative_wealth']>1+1e-8,'close_and_available_point_MaxDD':close_dd>=-30 and sampled_dd>=-30,'CVaR5_no_worse_SOXX':source_metrics['cvar5_pct']>=source_metrics['benchmark_cvar5_pct'],'minimum_two_complete_years':len(complete)>=2,'complete_years_70pct':len(complete)>=2 and sum(x['relative_wealth']>1 for x in complete)/len(complete)>=.70,'rolling126_70pct':positive>=need,'SOXX_block_lower_positive':soxx_ci['lower_bound_positive']}
coarse_pass=bool(all(gates.values()))
result={'decision':'conditional_Stage1_common_return_dedup_0bp_SOXX_coarse_pass_needs_2_5bp_forward' if coarse_pass else 'not_promoted_fixed_Stage1_common_return_dedup_0bp_failed','plan_sha256':sha(card/'plan.json'),'analysis_repair_plan_sha256':sha(card/'analysis_repair_plan.json'),'source_Recorder_status_before_repair':'FAILED','A0_exact_path_parity':True,'B_saved_prediction_parity':True,'new_model_fits':2,'new_A0_parity_paths':1,'new_candidate_paths':2,'new_native_accounts':2,'new_bootstraps':2,'candidate_SOXX_relative_wealth':float(source_metrics['relative_wealth']),'candidate_close_MaxDD_pct':close_dd,'candidate_sampled_MaxDD_pct':sampled_dd,'candidate_CVaR5_pct':float(source_metrics['cvar5_pct']),'SOXX_CVaR5_pct':float(source_metrics['benchmark_cvar5_pct']),'all_years':annual,'all_quarters':quarterly,'rolling126_windows':len(rolling),'rolling126_positive':positive,'rolling126_required':need,'SOXX_bootstrap':soxx_ci,'matched_B_C_relative_wealth':float(matched['relative_wealth']),'matched_B_C_bootstrap':matched_ci,'gates':gates,'coarse_SOXX_0bp_hard_gate_pass':coarse_pass,'model_fits_in_this_repair':0,'accounts_in_this_repair':0}
(card/'analysis_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder'),recorder_id=rid,resume=True):
 rec=R.get_recorder();assert rec.id==rid
 rec.save_objects(artifact_path='analysis',metrics=relative,annual_metrics=pd.DataFrame(annual),quarterly_metrics=pd.DataFrame(quarterly),rolling126=rolling,SOXX_uncertainty=soxx_ci,matched_B_C=matched,matched_B_C_uncertainty=matched_ci,gates=gates,result=result)
 rec.log_metrics(analysis_repair_only=1,candidate_SOXX_relative_wealth=result['candidate_SOXX_relative_wealth'],rolling126_positive=positive,coarse_SOXX_0bp_hard_gate_pass=int(coarse_pass))
result['source_Recorder_status_after_repair']='FINISHED';(card/'analysis_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
state['status']='finished';state['bootstrap_attempts']=2;(card/'run_state.json').write_text(json.dumps(state,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:v for k,v in result.items() if k not in ('all_years','all_quarters','SOXX_bootstrap','matched_B_C_bootstrap')},ensure_ascii=False,indent=2));print('annual',annual);print('SOXX_bootstrap_lower',soxx_ci['daily_lower_bound'])


{
  "decision": "not_promoted_fixed_Stage1_common_return_dedup_0bp_failed",
  "plan_sha256": "d33c5ea855bd06fb8a41911b6bace14b8ab0a0a0911db7325f5b730c96399d14",
  "analysis_repair_plan_sha256": "6a20cb52ad96264ad911fba548fde015f59e479d093a7ad3b815290916ed69b7",
  "source_Recorder_status_before_repair": "FAILED",
  "A0_exact_path_parity": true,
  "B_saved_prediction_parity": true,
  "new_model_fits": 2,
  "new_A0_parity_paths": 1,
  "new_candidate_paths": 2,
  "new_native_accounts": 2,
  "new_bootstraps": 2,
  "candidate_SOXX_relative_wealth": 0.9614726395822718,
  "candidate_close_MaxDD_pct": -28.599019139177926,
  "candidate_sampled_MaxDD_pct": -28.81039746032282,
  "candidate_CVaR5_pct": -4.712155261961044,
  "SOXX_CVaR5_pct": -4.90478329004924,
  "rolling126_windows": 819,
  "rolling126_positive": 369,
  "rolling126_required": 574,
  "matched_B_C_relative_wealth": 1.377269142169212,
  "gates": {
    "SOXX_wealth": false,
    "close_and_available_point_MaxDD": true,
    "CVaR5_no_wor

## 中文结论

原 A_0 路径精确复现，匹配 B 预测通过保存值校验，去重 C 与 B 各运行独立 0bp Tiered 账户。C/SOXX 财富比0.96147，回撤与CVaR粗门过，完整年仅2024赢，126日窗口369/819低于574，SOXX区块下界−0.00074884，固定版本不晋级。两拟合、两新账户；分析器三成本限制在账户保存后机械修复，未重训／重放。Recorder 与 NAS7项冷读一致。